# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from google.colab import userdata
from openai import OpenAI


OPENAI_API_KEY = userdata.get("OPENAI_API_KEY").strip()


client = OpenAI(api_key=OPENAI_API_KEY)

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [3]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ['customers', 'orders', 'products'],
    'definition': [
        'customer information such as customer_id, name, city, country',
        'order information such as order_id, customer_id, product_id, quantity, order_date',
        'product information such as product_id, product_name, category, price'
    ]
}

df = pd.DataFrame(data)
print(df)

       table                                         definition
0  customers  customer information such as customer_id, name...
1     orders  order information such as order_id, customer_i...
2   products  product information such as product_id, produc...


In [6]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [7]:
print(text_tables)

customers: customer information such as customer_id, name, city, country
orders: order information such as order_id, customer_id, product_id, quantity, order_date
products: product information such as product_id, product_name, category, price


In [8]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [9]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to find all orders made by customers in Germany?"
)

In [10]:
print(return_OAI(pqt1))

{
    "tables": ["customers", "orders"]
}


In [11]:
pqt3 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to find the total revenue by product category?"
)

In [12]:
print(return_OAI(pqt3))

```json
{
    "tables": ["orders", "products"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [13]:
pqt4 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to find all customers from France who ordered products in the 'Electronics' category?"
)

print(return_OAI(pqt4))

{
    "tables": ["customers", "orders", "products"]
}


In [14]:
pqt5 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to calculate the total revenue generated by each product?"
)

print(return_OAI(pqt5))

{
  "tables": ["orders", "products"]
}


In [15]:
pqt6 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to find customers who have never placed an order?"
)

print(return_OAI(pqt6))

{
    "tables": ["customers", "orders"]
}


In [16]:
pqt7 = prompt_question_tables.format(
    tables=text_tables,
    question="Which tables are needed to identify the most valuable customers?"
)

print(return_OAI(pqt7))

{
    "tables": ["customers", "orders", "products"]
}


In this lab I created different prompts to test how well the model can identify the correct tables based on a question.

First, I defined the tables and their definitions. This is important because the model only uses this information to understand the database structure.

Then I created prompts using prompt_question_tables. In each prompt I inserted a different question to see how the model reacts.

In the first attempts, I used clear and specific questions (for example customers from a country or revenue by product). These worked well. The model correctly selected the needed tables like customers, orders, and products.

Then I tried more complex questions like customers who never ordered something. This also worked, but required more reasoning from the model.

Finally, I tested a more vague question like “most valuable customers”. Here the model was less precise because the term is not clearly defined. This shows that unclear questions can lead to weaker results.

Overall, I learned that the quality of the result depends strongly on how clear the question and table definitions are. More precise prompts give better and more reliable answers.